# C1 classical: full run on a Colab T4

Kernel: **Colab, T4 GPU** (Runtime → Change runtime type → T4). Run all cells top to bottom.

Re-runs every tuning stage (s1–s4, 243 configs × seeds 0–4) and the `best` / `fast` configs on the report seeds (10–19), all tagged `hw=colab-t4`, so every time in the report comes from the same GPU type as the rest of the team. Expect roughly 30–60 min (Colab has only 2 CPU cores).

**Resumable:** rows are logged to a copy of `results/massimo.csv` on your Google Drive, so they survive a disconnect or a runtime reset. Just run all cells again: rows already logged for `hw=colab-t4` are skipped. The last code cell merges the Drive copy back into the repo.

If you re-run after a change, **restart the kernel first** (`common.data` only reads `COMP433_DATA` once, at import).

In [ ]:
# 1. Get the code onto the Colab machine
import os
import shutil
from getpass import getpass

REPO = "/content/repo"
BRANCH = "massimo/c1"

if not os.path.exists(REPO):
    token = getpass("GitHub token: ")
    !git clone -q -b {BRANCH} https://{token}@github.com/MassPaol/comp433-final-project.git {REPO}
    del token
else:
    !git -C {REPO} pull -q

%cd {REPO}
!pip install -q -e . -r requirements.txt

# Commit identity for saving results at the end (a fresh Colab VM has none)
!git -C {REPO} config user.name "Massimo Paolini"
!git -C {REPO} config user.email "mpaolini119@gmail.com"

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Copy CIFAR-10 from the team folder to the Colab machine (faster than reading from Drive)
DRIVE_DATA = "/content/drive/MyDrive/comp433_cache/data"  # needs the comp433_cache shortcut in My Drive
LOCAL_DATA = "/content/data"

if not os.path.exists(f"{LOCAL_DATA}/cifar-10-python.tar.gz"):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    shutil.copy(f"{DRIVE_DATA}/cifar-10-python.tar.gz", LOCAL_DATA)  # torchvision unpacks it on first load

os.environ["COMP433_DATA"] = LOCAL_DATA

In [ ]:
# 4. Import the project code (must come after COMP433_DATA is set) and check the data loads
import torch
from common import harness, data

assert data.DATA_ROOT == os.environ["COMP433_DATA"], (
    "common.data was imported before COMP433_DATA was set. Restart the kernel and run all cells again."
)
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "none (running on CPU)")

X, y, Xv, yv, classes = data.load_instance(0)
print(X.shape, Xv.shape, [data.CLASS_NAMES[c] for c in classes])  # expect (100, 32, 32, 3) (700, 32, 32, 3)

In [ ]:
# 5. Make sure this really is a T4 (every row gets tagged hw=colab-t4), and log to Drive
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(), (
    f"Expected a T4 GPU, got {torch.cuda.get_device_name() if torch.cuda.is_available() else 'no GPU'}")
HW = "colab-t4"

RESULTS_DIR = "/content/drive/MyDrive/comp433_c1_classical_results"
RESULTS = f"{RESULTS_DIR}/massimo.csv"
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.exists(RESULTS):
    shutil.copy("results/massimo.csv", RESULTS)  # start from the repo's rows
print("logging to", RESULTS)

In [ ]:
# 6. All tuning stages on seeds 0-4 (prints one line per config)
for stage in ["s1", "s2", "s3", "s4"]:
    !python -W ignore -m c1.classical.run --stage {stage} --seeds tune --hw {HW} --results-dir {RESULTS_DIR}

In [ ]:
# 7. Re-apply the selection rule with T4 times, and check it agrees with FINAL in configs.py
!python -m c1.classical.select --hw {HW} --results {RESULTS}

import pandas as pd
from c1.classical import configs, select as sel

summary = sel.summarize_configs(pd.read_csv(RESULTS), data.TUNE_SEEDS, HW)
summary = summary[summary.index.str.match(r"s[1-4]_")]
assert len(summary) == 243, f"expected 243 complete tuning configs, found {len(summary)}"
best, fast = sel.select(summary)
picked = {"best": best, "fast": fast}
if picked != configs.FINAL_SOURCE:
    raise RuntimeError(
        f"T4 timings pick {picked}, but configs.FINAL_SOURCE is {configs.FINAL_SOURCE}. "
        "Update FINAL_SOURCE in c1/classical/configs.py (and the selection entry in NOTES.md), "
        "push, restart the kernel and run all cells again.")
print("T4 selection agrees with configs.FINAL_SOURCE:", picked)

In [ ]:
# 8. best and fast on the report seeds 10-19
!python -W ignore -m c1.classical.run --config best fast --seeds report --hw {HW} --results-dir {RESULTS_DIR}

In [ ]:
# 9. Report-seed summary (these are the numbers for the report)
from scipy.stats import wilcoxon
from c1.classical.run import hw_of, own_rows

df = own_rows(pd.read_csv(RESULTS))
r = df[df.config_id.isin(["best", "fast"]) & df.seed.isin(data.REPORT_SEEDS) & (df.notes.map(hw_of) == HW)]
w = r.pivot_table(index=["seed", "class_pair"], columns="config_id",
                  values=["val_acc", "train_time_s", "infer_time_s"])
for c in ["best", "fast"]:
    acc = w["val_acc"][c] * 100
    infer = w["infer_time_s"][c].mean()
    print(f"{c}: {acc.mean():.2f} +/- {acc.std(ddof=0):.2f} %   train {w['train_time_s'][c].mean():.3f} s   "
          f"inference {infer:.3f} s / 700 img ({infer / 700 * 1000:.2f} ms/img)")
d = (w["val_acc"]["best"] - w["val_acc"]["fast"]) * 100
print(f"paired best - fast: {d.mean():+.2f} pp, best wins {(d > 0).sum()}/{len(d)}, "
      f"Wilcoxon p = {wilcoxon(d).pvalue:.4f}")

In [ ]:
# 10. Regenerate the report figures from the T4 rows
!python -m c1.classical.figures --hw {HW} --results {RESULTS}

In [ ]:
# 11. Merge the Drive copy back into the repo's results file
from c1.classical.run import merge_results
print("added", merge_results(RESULTS, "results/massimo.csv"), "rows to results/massimo.csv")

### 12. Save the results

Commit the new rows and figures from this machine, then **clear this notebook's outputs** before committing it (Edit → Clear all outputs):

```
!git add results/massimo.csv report/figures/c1_classical_*
!git commit -m "Run C1 classical on Colab T4"
!git push
```